### Project: The Intelligent Web Summarizer

We'll use a utility script to fetch and clean website content. Let's see it in action.

In [ ]:
from web_scraper import scrape_text_content, scrape_hyperlinks

sample_url = "https://en.wikipedia.org/wiki/Artificial_intelligence"
scraped_text = scrape_text_content(sample_url)

print(scraped_text)

### Engineering the Prompt

Effective LLM interaction relies on two primary roles:

1.  **System Prompt:** Sets the persona, context, and constraints (the "Who").
2.  **User Prompt:** Provides the specific task or data (the "What").

In [5]:
SYSTEM_PROMPT = """
You are an expert Information Analyst. Your mission is to transform raw, noisy website content 
into a high-impact executive summary. 

Follow these strict structural guidelines:
1. **Executive Summary**: A 2-sentence 'big picture' overview.
2. **Key Insights**: 3-5 high-value bullet points.
3. **Latest Developments**: Highlight any news, dates, or announcements.
4. **Tone & Sentiment**: Briefly describe the author's tone.

Filter out all navigation, headers, and footers. Respond in professional Markdown.
"""

USER_PROMPT_PREFIX = """
Please analyze the following scraped content and provide a structured report.
If the content is technical, ensure the summary remains accessible but accurate.
If no significant news is found, focus on the core value proposition of the site.

CONTENT:
"""

In [6]:
from typing import List, Dict
from web_scraper import scrape_text_content, scrape_hyperlinks
from openai import OpenAI
from IPython.display import Markdown, display
from dotenv import load_dotenv

env_loaded = load_dotenv(override=True)

def create_messages(content: str) -> List[Dict[str, str]]:
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"{USER_PROMPT_PREFIX}{content}"}
    ]

def summarize(url: str) -> str:
    content = scrape_text_content(url)
    openai_client = OpenAI()
    response = openai_client.chat.completions.create(
        model="gpt-5-mini",
        messages=create_messages(content)
    )
    return response.choices[0].message.content

def display_summary(url: str):
    print(f"Summarizing: {url}...")
    summary = summarize(url)
    display(Markdown(summary))

display_summary("https://en.wikipedia.org/wiki/Artificial_intelligence")

Summarizing: https://en.wikipedia.org/wiki/Artificial_intelligence...


Executive Summary
- The scraped Wikipedia article provides a comprehensive, encyclopedic overview of artificial intelligence (AI): its goals (learning, reasoning, perception, planning), major technical approaches (deep learning, neural networks, probabilistic methods, logic, GPT-style transformers), and wide-ranging applications across industry, science and society.  
- The entry emphasizes rapid recent progress (GPU-accelerated deep learning and transformer-based LLMs), large-scale deployment (generative AI, agents, search, healthcare, finance, military), and equally significant ethical, legal and environmental risks.

Key Insights
- Scope and goals: AI is framed as the study and engineering of systems that perform tasks associated with human intelligence (learning, reasoning, perception, decision-making), from narrow applications to the aspirational goal of artificial general intelligence (AGI).
- Core techniques driving progress: breakthroughs are attributed to neural networks/deep learning, GPU acceleration (circa 2012), and the transformer architecture (since ~2017); GPT-style LLMs are now central to many language and multimodal capabilities but remain prone to hallucinations.
- Broad, material applications: AI is embedded across search, recommendation systems, virtual assistants, autonomous vehicles, healthcare (e.g., AlphaFold 2), generative creative tools (text/image/video), finance, gaming, and scientific discovery.
- Risks and systemic issues: the article highlights privacy and copyright concerns, algorithmic bias and lack of explainability, concentration of power in Big Tech, misuse (deepfakes, surveillance, military), and major energy/power demands from AI data centers.
- Ongoing technical limits: despite impressive empirical gains, important problems remain—efficient reasoning, commonsense knowledge, reliable mathematical reasoning, and safe alignment—so many systems still produce errors or misleading outputs.

Latest Developments (dates and announcements called out in the scraped text)
- Feb 2023: Microsoft introduced Copilot Search (Bing Chat) as an AI-powered search/summary feature integrated into Edge and Bing.  
- 2023: Reports that AI-guided drug-discovery found a new antibiotic class effective against drug-resistant bacteria.  
- 2024: International Energy Agency — Electricity 2024 (analysis and forecast to 2026) highlighted rapidly growing power demand from data centers and AI; projections warned of doubling power demand for these uses by 2026.  
- Mar 2024: Amazon purchased a Pennsylvania nuclear-powered data center (reported) as part of addressing data-center energy needs.  
- 2024: Google DeepMind introduced SIMA, an agent capable of autonomously playing multiple previously unseen open-world video games and performing short tasks from natural-language instructions.  
- 2024: Machine-learning work accelerated Parkinson’s drug-screening (reporting a ten-fold speed-up and large cost reduction for early screening).  
- 2024 Indian elections: reported use of roughly US$50 million on authorized AI-generated campaign content, including deepfakes and localized translations.  
- Jan 2025 (as reported in the article): Microsoft proposed rStar-Math combining Monte Carlo tree search and stepwise reasoning to improve smaller models’ mathematical problem solving; other 2025 claims include Gemini Deep Think achieving IMO gold-medal results.  
Note: the above items are those mentioned in the scraped content; for confirmation and the latest status, consult primary sources.

Tone & Sentiment
- Neutral and encyclopedic: informational, descriptive and broadly balanced. The article conveys both enthusiasm for AI's capabilities and practical applications and caution about ethical, legal, safety and environmental risks.

## Rich Example: Cross-Model Financial Analysis

Let's raise the stakes. We'll use two different frontier models—**OpenAI (GPT-4o-mini)** and **DeepSeek (DeepSeek-Chat)**—to analyze the same financial data from Google Finance. 

This demonstrates how you can use multiple "expert opinions" to get a more balanced view of complex data like market trends.

In [7]:
from typing import List, Dict
from web_scraper import scrape_text_content, scrape_hyperlinks
from openai import OpenAI
import os
from IPython.display import Markdown, display
from dotenv import load_dotenv

env_loaded = load_dotenv(override=True)

def analyze_market_sentiment(url: str):
    # 1. Scrape the financial data
    market_data = scrape_text_content(url)

    prompt = f"Analyze the following market data and provide a professional summary of key movements:\n\n{market_data}"
    messages = [{"role": "user", "content": prompt}]
    
    # 2. Call OpenAI
    print("--- Analysis by OpenAI (GPT-5-mini) ---")
    oa_client = OpenAI()
    oa_res = oa_client.chat.completions.create(model="gpt-5-mini", messages=messages)
    display(Markdown(oa_res.choices[0].message.content))

    # 3. Call DeepSeek (OpenAI-compatible)
    print("\n--- Analysis by DeepSeek (DeepSeek-Chat) ---")
    ds_client = OpenAI(
        base_url="https://api.deepseek.com/v1", 
        api_key=os.getenv("DEEPSEEK_API_KEY")
    )

    try:
        ds_res = ds_client.chat.completions.create(model="deepseek-chat", messages=messages)
        display(Markdown(ds_res.choices[0].message.content))
    except Exception as e:
        print(f"DeepSeek Error: {e} (Check if your DEEPSEEK_API_KEY is valid and has balance)")
    
analyze_market_sentiment("https://www.google.com/finance/markets/indexes")

--- Analysis by OpenAI (GPT-5-mini) ---


Summary of key movements (based on the data provided)

Market-wide picture
- Global tape is mixed: US large-caps are essentially flat (S&P 500 +0.05%, Dow +0.10%), while the Nasdaq is modestly lower (~-0.2%). Small caps outperformed (Russell 2000 +1.18%).
- Europe showed a mixed session (DAX +0.25%, FTSE +0.42%, CAC and EURO STOXX modestly lower; IBEX notably weaker).
- Asia/India underperformed: major Asian indices fell (Nikkei -1.21%, SSE -1.26%, Hang Seng -1.72%), and Indian benchmarks dropped sharply (NIFTY 50 -1.30%, SENSEX -1.25%).

India-specific highlights
- NIFTY 50 at 25,471.10, down ~336 points (-1.30%); SENSEX at 82,626.76, down ~1,048 points (-1.25%).
- Sector movers: Nifty IT underperformed (~-1.44%); Nifty Bank down ~-0.91%. Several large-cap names listed (Reliance, HDFC Bank, BSE Ltd, M&M, ITC, SBI) showed modest intraday gains, indicating idiosyncratic stock-level strength despite index weakness.
- Headlines point to heavy two-day losses and commentary that foreign institutional flows and global cues are key near-term drivers.

Other notable points
- Volatility ticked up: VIX ~20.6 (+~1%), suggesting modestly higher investor nervousness.
- Canada’s TSX outperformed (+1.87%), while Brazil’s main index showed weakness.
- There are stock-specific extremes among most-active names (e.g., Engineers India and Physicswallah strong; some smaller names showing double-digit moves).

Interpretation and near-term implications
- The pattern looks like risk-off / profit-taking in Asia and India following recent rallies, while US indices held up relatively well (large-cap defensive bias) and small caps found buying.
- Rising volatility and reports of large near-term losses in India suggest sensitivity to FII flows and global macro headlines—watch liquidity and flows into Indian equities.
- Market participants should watch: upcoming macro calendar (rates/data), continued FII/DII flows, and technical support levels for Nifty/Sensex. Sector focus: defensive large caps and select stocks that are reporting positive company-specific news may outperform in the near term.

Data note
- The source contains some duplicated and slightly inconsistent item listings (e.g., mixed sign conventions on a few names). Verify live market quotes before making any trading decisions.


--- Analysis by DeepSeek (DeepSeek-Chat) ---


Based on the provided market data, here is a professional summary of the key movements and trends.

### **Executive Summary: A Day of Divergence**
Global markets exhibited a clear **East-West divergence** on this trading day. Major **Asian and Indian indices sold off sharply**, while **US and European markets were mostly flat to positive**. This suggests localized or regional concerns are driving volatility in Asia, contrasting with relative stability in the West.

---

### **Key Market Movements**

**1. Indian Markets: Significant Correction**
*   **NIFTY 50:** 25,471.10, **-1.30%** (-336.10 pts)
*   **SENSEX:** 82,626.76, **-1.25%** (-1,048.16 pts)
*   **Nifty Bank:** -0.91%
*   **Nifty IT:** -1.44%

*   **Analysis:** Indian benchmarks led the declines, with selling pressure broad-based across sectors. The news headlines ("Stock market crash," "market bloodbath") point to significant investor concern, likely driven by **Foreign Institutional Investor (FII) outflows, negative global cues from Asia, and potential profit-taking** after recent rallies. The underperformance of the IT index suggests weakness in the tech sector, often sensitive to global growth outlooks.

**2. Asia-Pacific: Broad-Based Weakness**
*   **Nikkei 225 (Japan):** -1.21%
*   **SSE Composite (China):** -1.26%
*   **Hang Seng (Hong Kong):** -1.72% (the worst performer shown)
*   **S&P Asia 50:** A slight outlier, up +0.13%.

*   **Analysis:** The sell-off was regional, not isolated to India. Notably, the Nikkei's decline comes despite a recent post-election rally, indicating a potential consolidation or reversal. Chinese and Hong Kong markets continued to face headwinds.

**3. Americas & Europe: Resilience and Mixed Performance**
*   **US Markets:** Showed remarkable stability. The **S&P 500 (+0.05%)** and **Dow Jones (+0.099%)** were marginally positive, while the **Nasdaq Composite (-0.22%)** saw a slight tech-led dip. The **Russell 2000 (+1.18%)** outperformed, indicating strength in smaller domestic companies.
*   **Canada (TSX):** Significantly outperformed, up **+1.87%**.
*   **Europe:** Mixed but generally stable. The **DAX (Germany, +0.25%)** and **FTSE 100 (UK, +0.42%)** gained, while the **CAC 40 (France, -0.35%)** and **IBEX 35 (Spain, -1.25%)** declined.

**4. Volatility & Risk Gauge**
*   **India VIX:** At 20.60, up +1.06%. This elevated level (above 20) confirms **heightened fear and expected near-term volatility** in the Indian market, aligning with the sharp index drops.

**5. Notable Individual Stock Movements (India)**
*   **Major Decliners:** Reliance Industries (-2.09%), HDFC Bank (-1.81%), M&M (-1.67%), ITC (-1.21%).
*   **Major Gainer:** Engineers India Ltd surged **+10.18%**, indicating strong stock-specific interest.
*   **Most Active:** Vodafone Idea was the most traded stock, closing down **-2.34%**.

---

### **Synthesis and Implied Narrative**

*   **Primary Driver:** The data points to **region-specific risk-off sentiment in Asia**. The simultaneous drop in major Asian indices suggests a common catalyst, such as concerns about regional economic growth, currency movements, or geopolitical tensions.
*   **Market Isolation:** The weakness was **not global**. The stability in US and key European markets indicates that the selling pressure was contained and not driven by a systemic global risk event (e.g., a major shift in US Fed policy or a crisis).
*   **Indian Context:** India's underperformance appears to be part of this broader Asian trend, exacerbated by **local factors** such as FII selling and profit-booking after a strong run. The news snippets explicitly link the fall to FII flows and global cues.
*   **Sector Story:** The underperformance of **Nifty IT** globally (and Nasdaq's slight dip) hints at some caution toward the technology sector.

### **Conclusion**
The trading session was characterized by a **sharp dichotomy**: a **risk-off environment in Asia** contrasting with a **neutral-to-slightly-positive environment in the West**. Investors rotated away from Asian equities, with Indian markets experiencing a pronounced correction amid elevated volatility. The next moves will likely depend on the persistence of FII outflows for India and whether the negative sentiment in Asia finds a floor or spreads to other regions.

**Disclaimer:** This analysis is based on a single snapshot of market data. For investment decisions, deeper analysis of fundamentals, macroeconomic factors, and ongoing news flow is essential.